### Lab 4.2: Trích xuất và Mô tả Đặc trưng Bất biến SIFT & SURF

#### Phần 0: Khởi tạo Notebook và Load Dữ liệu

Tương tự như Lab 4.1, chúng ta cần import các thư viện cốt lõi và tải về một số cặp ảnh mẫu. Để thấy được sức mạnh của SIFT/SURF, ta sẽ dùng 2 bức ảnh chụp cùng một vật thể nhưng ở góc độ, khoảng cách (tỷ lệ) và điều kiện ánh sáng khác nhau.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from os.path import join

# Khai báo lại hàm phụ trợ hiển thị ảnh
def imshow_cv(title, img, figsize=(8, 6)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 3:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

def imshow_multiple(title, images, titles, figsize=(16, 8)):
    fig, axs = plt.subplots(1, len(images), figsize=figsize)
    fig.suptitle(title, fontsize=16)
    for i, (img, t) in enumerate(zip(images, titles)):
        if len(img.shape) == 3:
            axs[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axs[i].imshow(img, cmap='gray')
        axs[i].set_title(t)
        axs[i].axis('off')
    plt.show()

# Tạo thư mục làm việc cho Lab 4.2
lab_path = "/content/CV_Labs/Lab_4_2/"
os.makedirs(lab_path, exist_ok=True)
%cd "{lab_path}"

# Tải cặp ảnh mẫu chứa cùng một đối tượng (ví dụ: biển báo giao thông hoặc ảnh sách)
# để test tính bất biến của đặc trưng
!wget -q -O target_object.jpg "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/box.png"
!wget -q -O scene_image.jpg "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/box_in_scene.png"

img_target = cv2.imread('target_object.jpg')
img_scene = cv2.imread('scene_image.jpg')

# Hiển thị ảnh mẫu
imshow_multiple('Dữ liệu đầu vào', [img_target, img_scene], ['Ảnh mẫu (Target)', 'Ảnh bối cảnh (Scene)'])

#### Phần 1: Thuật toán SIFT ("Tiêu chuẩn vàng")

**Mục tiêu:** Thuật toán SIFT (Scale-Invariant Feature Transform) do David Lowe đề xuất là một trong những thuật toán mô tả đặc trưng mạnh mẽ nhất. Sinh viên sẽ thực hành trích xuất điểm đặc trưng (Keypoint) trải qua 4 bước: Dò tìm cực trị không gian tỷ lệ (Scale-space extrema), Định vị điểm, Gán hướng và Tạo vector mô tả 128 chiều.

**1.1 Dò tìm điểm đặc trưng (Keypoint Detection)**

OpenCV cung cấp sẵn lớp `cv2.SIFT_create()`. Chúng ta sẽ trích xuất keypoint và vẽ chúng lên ảnh. Việc sử dụng cờ `cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS` sẽ giúp vẽ các vòng tròn thể hiện cả **kích thước (scale)** và **hướng (orientation)** của từng điểm đặc trưng.

In [ ]:
# Khởi tạo đối tượng SIFT
sift = cv2.SIFT_create()

# Chuyển ảnh sang thang độ xám (SIFT hoạt động trên ảnh xám)
gray_target = cv2.cvtColor(img_target, cv2.COLOR_BGR2GRAY)
gray_scene = cv2.cvtColor(img_scene, cv2.COLOR_BGR2GRAY)

# Tìm Keypoints và tính toán Descriptor cùng lúc
# Hàm trả về 2 biến: keypoints (chứa tọa độ, kích thước, góc) và descriptors (vector đặc trưng)
kp_target, des_target = sift.detectAndCompute(gray_target, None)
kp_scene, des_scene = sift.detectAndCompute(gray_scene, None)

print(f"Số lượng Keypoint trên ảnh Target: {len(kp_target)}")
print(f"Số lượng Keypoint trên ảnh Scene: {len(kp_scene)}")

# Vẽ keypoints với thông tin về Tỷ lệ và Hướng (vòng tròn và đường bán kính)
img_target_sift = cv2.drawKeypoints(img_target, kp_target, None,
                                    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img_scene_sift = cv2.drawKeypoints(img_scene, kp_scene, None,
                                   flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

imshow_multiple('Điểm đặc trưng SIFT (RICH_KEYPOINTS)',
                [img_target_sift, img_scene_sift],
                ['SIFT trên Target', 'SIFT trên Scene'])

**1.2 Phân tích cấu trúc Descriptor của SIFT**

SIFT tạo ra bộ mô tả dựa trên lưới lân cận $16 \times 16$ pixel xung quanh keypoint, chia thành 16 ô (mỗi ô $4 \times 4$). Trong mỗi ô, thuật toán tính histogram gồm 8 hướng gradient. Do đó, vector kết quả có chiều dài là $16 \times 8 = 128$ chiều.

In [ ]:
# Phân tích Descriptor của ảnh Target
print("Kích thước ma trận Descriptor của ảnh Target:", des_target.shape)

# Trích xuất thử thông tin của Keypoint đầu tiên
first_kp = kp_target
first_des = des_target

print(f"\nThông tin Keypoint #0:")
print(f" - Tọa độ (x, y): {first_kp.pt}")
print(f" - Tỷ lệ (kích thước vùng ảnh hưởng - size): {first_kp.size:.2f}")
print(f" - Hướng chính (angle): {first_kp.angle:.2f} độ")

print(f"\nVector mô tả SIFT của Keypoint #0 (128 chiều):")
print(first_des)

# (Tùy chọn) Hiển thị trực quan Descriptor dưới dạng bản đồ nhiệt (Heatmap)
plt.figure(figsize=(10, 4))
plt.imshow(des_target[:20].T, cmap='viridis', aspect='auto') # Hiển thị 20 vector đầu tiên
plt.title('Bản đồ nhiệt 20 SIFT Descriptor đầu tiên (128 chiều)')
plt.xlabel('Chỉ số Keypoint')
plt.ylabel('Thành phần của Vector (0-127)')
plt.colorbar(label='Độ lớn')
plt.show()

*Gợi ý thảo luận cho sinh viên:* Hãy quan sát bản đồ nhiệt để thấy rằng mỗi điểm keypoint (mỗi cột) đều sở hữu một cấu trúc 128 con số hoàn toàn khác biệt. Điều này giải thích tại sao SIFT được gọi là "tiêu chuẩn vàng" về độ độc nhất.

### Phần 2: Thuật toán SURF (Speeded Up Robust Features - Tối ưu tốc độ)

**Mục tiêu:** Mặc dù SIFT là "tiêu chuẩn vàng" về độ chính xác, nhưng quy trình tích chập Gaussian của nó lại rất tốn kém tài nguyên tính toán. Thuật toán SURF ra đời nhằm giải quyết bài toán tốc độ bằng cách sử dụng **Ảnh tích phân (Integral Images)** và **Bộ lọc Box (Box Filters)** để xấp xỉ đạo hàm, giúp tăng tốc độ xử lý lên gấp 3-10 lần so với SIFT.

#### 2.1 Khởi tạo SURF và Tinh chỉnh Ngưỡng Hessian

SURF dựa trên định thức của ma trận Hessian để phát hiện các điểm đốm (blob). Điểm khác biệt quan trọng khi lập trình SURF là việc thiết lập tham số `hessianThreshold`. Ngưỡng này càng cao thì số lượng điểm đặc trưng giữ lại càng ít, nhưng đổi lại chất lượng và độ ổn định của chúng sẽ càng cao.

*Lưu ý cho sinh viên:* Vì SURF từng là thuật toán có bản quyền, trong một số phiên bản OpenCV, nó được đặt trong module `xfeatures2d`.

In [ ]:
# Khởi tạo thuật toán SURF với ngưỡng Hessian ban đầu là 400
# (Nếu báo lỗi thư viện, hãy đảm bảo bạn cài đặt opencv-contrib-python phiên bản tương thích)
surf_400 = cv2.xfeatures2d.SURF_create(hessianThreshold=400)

# Khởi tạo một đối tượng SURF khác với ngưỡng cao hơn để so sánh
surf_5000 = cv2.xfeatures2d.SURF_create(hessianThreshold=5000)

# Trích xuất Keypoint và tính toán Descriptor
kp_surf400, des_surf400 = surf_400.detectAndCompute(gray_scene, None)
kp_surf5000, des_surf5000 = surf_5000.detectAndCompute(gray_scene, None)

print(f"Số lượng Keypoint với Hessian=400: {len(kp_surf400)}")
print(f"Số lượng Keypoint với Hessian=5000: {len(kp_surf5000)}")

# Vẽ keypoint lên ảnh để trực quan hóa
img_surf400 = cv2.drawKeypoints(img_scene, kp_surf400, None,
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
img_surf5000 = cv2.drawKeypoints(img_scene, kp_surf5000, None,
                                 flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

imshow_multiple('So sánh phân ngưỡng Hessian trong SURF',
                [img_surf400, img_surf5000],
                ['Hessian Threshold = 400 (Nhiều điểm)', 'Hessian Threshold = 5000 (Ít & ổn định)'])

#### 2.2 Phân tích Vector Mô tả SURF (64 chiều)

Để tăng tốc độ tính toán cho các ứng dụng thời gian thực, SURF đã tối giản vector mô tả. Thay vì chia thành 16 ô với 8 hướng gradient như SIFT (tạo ra 128 chiều), SURF chia vùng ảnh thành $4 \times 4$ ô, nhưng bên trong mỗi ô chỉ tính toán 4 giá trị dựa trên **Haar-wavelet**: tổng đáp ứng theo hướng ngang ($\sum d_x$), tổng đáp ứng dọc ($\sum d_y$) và tổng giá trị tuyệt đối của chúng ($\sum |d_x|, \sum |d_y|$). Nhờ vậy, kích thước vector giảm đi một nửa.

In [ ]:
# Phân tích Descriptor của ảnh với Hessian=5000
print("Kích thước ma trận Descriptor SURF:", des_surf5000.shape)

if len(kp_surf5000) > 0:
    first_kp_surf = kp_surf5000
    first_des_surf = des_surf5000

    print(f"\nThông tin Keypoint #0 (SURF):")
    print(f" - Tọa độ (x, y): {first_kp_surf.pt}")
    print(f" - Tỷ lệ (size): {first_kp_surf.size:.2f}")
    print(f" - Điểm phản hồi Hessian (response): {first_kp_surf.response:.2f}")

    print(f"\nVector mô tả SURF (Chỉ có 64 chiều):")
    print(first_des_surf)

    # Hiển thị trực quan Descriptor SURF dưới dạng bản đồ nhiệt
    plt.figure(figsize=(8, 3))
    # Hiển thị tối đa 20 vector đầu tiên (nếu có đủ)
    num_vec = min(20, des_surf5000.shape)
    plt.imshow(des_surf5000[:num_vec].T, cmap='plasma', aspect='auto')
    plt.title('Bản đồ nhiệt SURF Descriptor (64 chiều)')
    plt.xlabel('Chỉ số Keypoint')
    plt.ylabel('Thành phần của Vector (0-63)')
    plt.colorbar(label='Độ lớn')
    plt.show()
else:
    print("Không tìm thấy điểm đặc trưng nào với ngưỡng Hessian hiện tại.")

*Gợi ý thảo luận:* Giảng viên có thể yêu cầu sinh viên chạy cả hai đoạn code SIFT (từ Phần 1) và SURF (Phần 2) bằng module `time` trong Python để đo lường và so sánh thực tế thời gian thực thi của hàm `detectAndCompute` trên những bức ảnh lớn.